# Kinthread STaR round on a free Colab T4
Our reader proposes readings (8 beam + 16 samples) for every keyed symgen question; a reading that reaches the key (and names every person, drops no sentence, at least two readings agree) becomes a training row; exact-reader readings are kept as distilled rows. Output: `star_r3.jsonl.gz` in the shared folder (resumable).

In [ ]:
import os, torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
REPO = '/content/Kinthread-llm-model'
if not os.path.exists(REPO):
    !git clone --depth 1 -b main https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
from google.colab import drive
drive.mount('/content/drive')
BASE = next(b for b in ('/content/drive/MyDrive/kinthread_shared', '/content/drive/MyDrive/kinthread') if os.path.isdir(b))
CKPT = next(f'{BASE}/runs/{r}/best.pt' for r in ('v10-r6a-rl', 'v10-r6a', 'v10-r5-plain') if os.path.exists(f'{BASE}/runs/{r}/best.pt'))
print('reader', CKPT)
os.makedirs('data/real_questions', exist_ok=True)
!tar -xzf {BASE}/real_questions.tgz -C data/real_questions 2>/dev/null

In [ ]:
!python star.py --ckpt {CKPT} --questions 'data/symgen/*.jsonl.gz' --out {BASE}/star_r3.jsonl.gz --beam 8 --samples 16 --bs 32 2>&1 | grep -v Warn | tail -5